# BCI Assignment code
## Daniils Krasovskis & Roy van Heertum

AI disclaimer: Generative AI was used to create sections of this code but everything was checked, and where necessary, corrected by the authors.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from pathlib import Path
import re
from scipy.io import loadmat
from scipy.signal import welch
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_validate

import mne

import ast

In [2]:
# =========================
# 3. PATHS
# =========================

PROJECT_ROOT = Path(r"C:\Users\royva\BCIdata")

DENS_ROOT = PROJECT_ROOT / "DENS_dataset"
PHYS_ROOT = PROJECT_ROOT / "Physiological Data"
RAW_ROOT = PHYS_ROOT / "Raw Data"
PREPROC_GROUP_ROOT = PHYS_ROOT / "PreProcessed-Emotion (Group-wise)"
PREPROC_SUBJECT_ROOT = PHYS_ROOT / "PreProcessed-Subject_Wise"
EEG_DATA_DIR = PREPROC_GROUP_ROOT / "EEG_Data"
BEH_ROOT = RAW_ROOT / "Behavioural Data"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DENS_ROOT exists:", DENS_ROOT.exists())
print("PHYS_ROOT exists:", PHYS_ROOT.exists())
print("RAW_ROOT exists:", RAW_ROOT.exists())
print("PREPROC_GROUP_ROOT exists:", PREPROC_GROUP_ROOT.exists())
print("PREPROC_SUBJECT_ROOT exists:", PREPROC_SUBJECT_ROOT.exists())
print("EEG_DATA_DIR exists:", EEG_DATA_DIR.exists())
print("BEH_ROOT exists:", BEH_ROOT.exists())

PROJECT_ROOT: C:\Users\royva\BCIdata
DENS_ROOT exists: True
PHYS_ROOT exists: True
RAW_ROOT exists: True
PREPROC_GROUP_ROOT exists: True
PREPROC_SUBJECT_ROOT exists: False
EEG_DATA_DIR exists: True
BEH_ROOT exists: True


In [3]:
# =========================
# FULL INSPECTION BLOCK
# =========================

# -------------------------
# FIND ALL PREPROCESSED .MAT FILES
# -------------------------
mat_files = sorted(EEG_DATA_DIR.rglob("*.mat"))
print("\n" + "=" * 70)
print("PREPROCESSED .MAT FILES")
print("=" * 70)
print(f"Found {len(mat_files)} .mat files")

for f in mat_files[:15]:
    print("-", f.name)

# -------------------------
# PARSE FILE NAMES
# Example:
# Amused_mit004Trial-3Click-1-slor.mat
# -------------------------

PROJECT_ROOT = Path(r"C:\Users\royva\BCIdata")
PHYS_ROOT = PROJECT_ROOT / "Physiological Data"
RAW_ROOT = PHYS_ROOT / "Raw Data"
BEH_ROOT = RAW_ROOT / "Behavioural Data"

beh_csvs = sorted(BEH_ROOT.rglob("*.csv"))
print(f"Found {len(beh_csvs)} behavioural CSV files")
pattern = re.compile(r"^(?P<emotion>.+?)_(?P<subject>mit[a-zA-Z0-9]+)Trial-(?P<trial>\d+)Click-(?P<click>\d+)-slor\.mat$")
parsed_rows = []

for f in mat_files:
    m = pattern.match(f.name)
    if m:
        parsed_rows.append({
            "filename": f.name,
            "file": str(f),
            "emotion": m.group("emotion"),
            "subject": m.group("subject"),
            "trial": int(m.group("trial")),
            "click": int(m.group("click")),
        })
    else:
        parsed_rows.append({
            "filename": f.name,
            "file": str(f),
            "emotion": None,
            "subject": None,
            "trial": None,
            "click": None,
        })

df_mat = pd.DataFrame(parsed_rows)

print("\n" + "=" * 70)
print("PARSED FILE INDEX")
print("=" * 70)
print(df_mat.head(10))
print("\nParsed successfully:", df_mat["subject"].notna().sum(), "/", len(df_mat))
print("Unique subjects in preprocessed EEG:", df_mat["subject"].nunique())

print("\nTop subject counts:")
print(df_mat["subject"].value_counts().head(15))

print("\nTop emotion counts:")
print(df_mat["emotion"].value_counts().head(15))

# -------------------------
# INSPECT ONE .MAT FILE
# -------------------------
if len(mat_files) > 0:
    example_mat = mat_files[0]
    mat_data = loadmat(example_mat)
    
    print("\n" + "=" * 70)
    print("ONE .MAT FILE INSPECTION")
    print("=" * 70)
    print("Example file:", example_mat.name)

    print("\nKeys inside .mat:")
    for k in mat_data.keys():
        print("-", k)

    print("\nShapes / numeric summaries:")
    for k, v in mat_data.items():
        if isinstance(v, np.ndarray):
            print(f"\n{k}")
            print("  shape:", v.shape)
            print("  dtype:", v.dtype)
            if v.size > 0 and np.issubdtype(v.dtype, np.number):
                print("  min :", np.min(v))
                print("  max :", np.max(v))
                print("  mean:", np.mean(v))
else:
    print("No .mat files found.")

# -------------------------
# 1. CLEAN BEHAVIOURAL DATA & ALIGN TRIAL INDEX
# -------------------------
beh_csvs = sorted(BEH_ROOT.rglob("*.csv"))
df_beh = pd.concat([pd.read_csv(f) for f in beh_csvs], ignore_index=True)

df_beh = df_beh.copy()
df_beh["participant"] = df_beh["participant"].astype(str)

# Identify the correct trial column automatically
trial_col = None
for c in ["trials.thisTrialN.1", "trials.thisTrialN"]:
    if c in df_beh.columns:
        trial_col = c
        break

if trial_col is None:
    raise ValueError("No trial column found in behavioural CSV.")

# Add 1 to align the 0-indexed CSV trials with the 1-indexed .mat filenames
df_beh["trial"] = pd.to_numeric(df_beh[trial_col], errors="coerce") + 1

df_beh_clean = df_beh[df_beh["participant"].notna() & df_beh["trial"].notna()].copy()
df_beh_clean["trial"] = df_beh_clean["trial"].astype(int)

# -------------------------
# 2. EXTRACT PER-CLICK RATINGS FROM 'Emotion_Name'
# -------------------------
def parse_click_ratings(row):
    """
    Parses the stringified dictionary in Emotion_Name.
    List positions: 0=HVHA, 1=LVHA, 2=LVLA, 3=HVLA, 4=Misc
    """
    emotion_str = row.get("Emotion_Name", "")
    if pd.isna(emotion_str) or not isinstance(emotion_str, str) or not emotion_str.startswith("{"):
        return []
    
    try:
        emo_dict = ast.literal_eval(emotion_str)
        click_rows = []
        
        for key_str, em_list in emo_dict.items():
            try:
                trial_click_idx = int(key_str) # Dict keys '0', '1', etc.
            except ValueError:
                continue
            
            valence_class = np.nan
            dict_emotion = "Unknown"
            
            # Identify which quadrant list index is populated for this click
            for idx, val in enumerate(em_list):
                if isinstance(val, str) and len(val.strip()) > 0:
                    if idx in [0, 3]: # 0: HVHA, 3: HVLA -> High Valence
                        valence_class = "high"
                    elif idx in [1, 2]: # 1: LVHA, 2: LVLA -> Low Valence
                        valence_class = "low"
                    
                    # Extract the pure emotion prefix for our validation print (e.g., 'Angry' from 'Angry: Feeling...')
                    dict_emotion = val.split(':')[0].strip()
                    break 
            
            click_rows.append({
                "participant": str(row["participant"]),
                "trial": int(row["trial"]),
                "trial_click_idx": trial_click_idx,
                "valence_class": valence_class,
                "dict_emotion": dict_emotion
            })
        return click_rows
    except Exception:
        return []

# Explode the per-trial CSV rows into per-click rows
all_clicks = []
for _, row in df_beh_clean.iterrows():
    all_clicks.extend(parse_click_ratings(row))

df_beh_clicks = pd.DataFrame(all_clicks)
# Drop miscellaneous/unclassified clicks
df_beh_clicks = df_beh_clicks.dropna(subset=["valence_class"]).copy()

print(f"Successfully extracted {len(df_beh_clicks)} valid click ratings from dictionaries.")

# -------------------------
# 3. PREPARE .MAT FILE INDEX & RANK CUMULATIVE CLICKS
# -------------------------
df_events = df_mat.copy()
df_events["participant"] = df_events["subject"]

# Convert cumulative clicks into 0-indexed trial click rankings
df_events = df_events.sort_values(by=["participant", "trial", "click"])
df_events["trial_click_idx"] = (
    df_events.groupby(["participant", "trial"])["click"]
    .rank(method="first")
    .astype(int) - 1  # -1 to make it 0-indexed to match dictionary keys
)

# -------------------------
# 4. MERGE EVENTS WITH PER-CLICK RATINGS
# -------------------------
df_merged = df_events.merge(
    df_beh_clicks,
    on=["participant", "trial", "trial_click_idx"],
    how="inner" # Inner join keeps only safely paired EEG windows and ratings
)

print("\n" + "=" * 75)
print("SANITY CHECK: FILENAME EMOTION vs DICTIONARY EMOTION ALIGNMENT")
print("=" * 75)
print("If 'emotion' and 'dict_emotion' match, the alignment configuration is flawless.")
print(df_merged[["participant", "trial", "click", "trial_click_idx", "emotion", "dict_emotion", "valence_class"]].head(20).to_string(index=False))

print("\nMerged rows (EEG events with valid quadrant labels):", len(df_merged))
print("\nFinal Valence Class Counts:")
print(df_merged["valence_class"].value_counts())

# Pass down to feature extraction and classification steps
df_val = df_merged.copy()

# -------------------------
# 5. LOAD SAMPLE CONFIRMATION
# -------------------------
if len(df_val) > 0:
    row = df_val.iloc[0]
    sample = loadmat(row["file"])["data"]
    print("\n One loaded sample inspection:")
    print("Filename:", row["filename"])
    print("Data Shape:", sample.shape)
    print("Assigned Valence Class:", row["valence_class"])
else:
    print("\n Warning: No matched data available.")


PREPROCESSED .MAT FILES
Found 475 .mat files
- Amused_mit004Trial-3Click-1-slor.mat
- Amused_mit014Trial-2Click-4-slor.mat
- Amused_mit014Trial-3Click-7-slor.mat
- Amused_mit023Trial-2Click-2-slor.mat
- Amused_mit052Trial-4Click-5-slor.mat
- Amused_mit052Trial-4Click-6-slor.mat
- Amused_mit052Trial-4Click-7-slor.mat
- Amused_mit072Trial-2Click-6-slor.mat
- Amused_mit079Trial-9Click-23-slor.mat
- Amused_mit082Trial-3Click-5-slor.mat
- Amused_mit099Trial-9Click-17-slor.mat
- Amused_mit113Trial-2Click-1-slor.mat
- Amused_mit114Trial-10Click-29-slor.mat
- Amused_mit122Trial-1Click-1-slor.mat
- Delighted_mit040Trial-4Click-3-slor.mat
Found 50 behavioural CSV files

PARSED FILE INDEX
                                filename  \
0   Amused_mit004Trial-3Click-1-slor.mat   
1   Amused_mit014Trial-2Click-4-slor.mat   
2   Amused_mit014Trial-3Click-7-slor.mat   
3   Amused_mit023Trial-2Click-2-slor.mat   
4   Amused_mit052Trial-4Click-5-slor.mat   
5   Amused_mit052Trial-4Click-6-slor.mat   
6   

In [9]:
# =============================================================================
# EEG VALENCE ANALYSIS PIPELINE
# Primary:   Univariate statistics (Mixed-Effects Models + Cohen's d) per band×region
# Secondary: Ablation classification (one band / one region at a time)
# =============================================================================

# =============================================================================
# 0. CONFIG 
# =============================================================================
 
# ── Shared style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        16,
    "axes.titlesize":   20,
    "axes.labelsize":   16,
    "xtick.labelsize":  16,
    "ytick.labelsize":  24,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "figure.dpi":       300,
})
 
SFREQ = 250

BANDS = {
    "theta": (4,  8),
    "alpha": (8,  13),
    "beta": (13, 30),
    "gamma": (30, 40),
}

REGIONS = {
    "prefrontal": ["E22", "E9", "E19", "E11"],       # Fp1, Fp2, Fpz, Fz
    "frontal": ["E24", "E124", "E33", "E122"],       # F3, F4, F7, F8
    "central": ["E36", "E104", "E129"],              # C3, C4, Cz
    "temporal": ["E45", "E108"],                     # T7, T8
    "parietal": ["E52", "E92", "E62", "E58", "E96"], # P3, P4, Pz, P7, P8
    "occipital": ["E70", "E83", "E75"]               # O1, O2, Oz
}

CHANNEL_NAMES = [f"E{i}" for i in range(1, 129)]

RESULTS_DIR = PROJECT_ROOT / "results_univariate"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# MNE info for topomaps
info = mne.create_info(ch_names=CHANNEL_NAMES, sfreq=SFREQ, ch_types="eeg")
montage = mne.channels.make_standard_montage("GSN-HydroCel-128")
info.set_montage(montage, match_case=False, on_missing="warn")

print("=" * 70)
print("CONFIG LOADED")
print("=" * 70)

# =============================================================================
# 1. BUILD TRIAL-LEVEL FEATURE MATRIX
# =============================================================================

print("\n[1] Extracting band×region features from .mat files ...")

df_work = df_merged[df_merged['valence_class'].isin(['low', 'high'])].copy().reset_index(drop=True)

print(f"Trials (clicks) filtered for analysis: {len(df_work)}")
print(f"Class balance:\n{df_work['valence_class'].value_counts().to_string()}")

def compute_band_region_power(eeg_data, sfreq, bands, regions, channel_names):
    """
    eeg_data : np.ndarray, shape (n_channels, n_times)
    Returns  : dict of {f"{band}_{region}": mean_log_power}
    """
    nperseg = min(int(sfreq * 2), eeg_data.shape[1])
    freqs, psd = welch(eeg_data, fs=sfreq, nperseg=nperseg, axis=-1)

    features = {}
    for band_name, (fmin, fmax) in bands.items():
        freq_mask = (freqs >= fmin) & (freqs <= fmax)
        for region_name, region_chs in regions.items():
            ch_idx = [channel_names.index(ch) for ch in region_chs if ch in channel_names]
            if not ch_idx or not np.any(freq_mask):
                features[f"{band_name}_{region_name}"] = np.nan
            else:
                region_psd = psd[np.ix_(ch_idx, freq_mask)]
                mean_power = region_psd.mean()
                features[f"{band_name}_{region_name}"] = np.log(mean_power + 1e-10)
    return features

rows = []
for _, row in df_work.iterrows():
    try:
        eeg = loadmat(row["file"])["data"]  # Shape: (128, n_timepoints)
        if eeg.shape[0] != 128:
            continue
        
        feats = compute_band_region_power(eeg, SFREQ, BANDS, REGIONS, CHANNEL_NAMES)
        
        feats.update({
            "filename": row.get("filename", Path(row["file"]).name),
            "participant": row["participant"],
            "trial": row["trial"],
            "click": row.get("click", row.get("trial_click_idx", np.nan)),
            "valence_class": row["valence_class"],
        })
        rows.append(feats)
    except Exception as e:
        pass  # Skip unreadable files silently

df_features = pd.DataFrame(rows)

meta_cols    = ["filename", "participant", "trial", "click", "valence_class"]
feature_cols = [c for c in df_features.columns if c not in meta_cols]

# Drop rows with any NaN feature
df_features = df_features.dropna(subset=feature_cols).reset_index(drop=True)

print(f"Final feature matrix: {df_features.shape[0]} rows x {len(feature_cols)} features")
df_features.to_csv(RESULTS_DIR / "trial_band_region_features.csv", index=False)
print(f"Saved: trial_band_region_features.csv")

# =============================================================================
# 2. PRIMARY ANALYSIS — UNIVARIATE STATISTICS PER BAND×REGION
# =============================================================================

print("\n[2] Running univariate statistics ...")

def cohens_d(a, b):
    """Pooled-SD Cohen's d (positive = a > b)."""
    n_a, n_b = len(a), len(b)
    pooled_std = np.sqrt(
        ((n_a - 1) * np.std(a, ddof=1)**2 + (n_b - 1) * np.std(b, ddof=1)**2)
        / (n_a + n_b - 2)
    )
    return (np.mean(a) - np.mean(b)) / (pooled_std + 1e-10)

low_trials  = df_features[df_features["valence_class"] == "low"]
high_trials = df_features[df_features["valence_class"] == "high"]

stat_rows = []
for feat in feature_cols:
    vals_low  = low_trials[feat].dropna().values
    vals_high = high_trials[feat].dropna().values
    d = cohens_d(vals_high, vals_low)

    df_tmp = df_features[["participant", "trial", "valence_class", feat]].copy()
    df_tmp["valence_bin"] = (df_tmp["valence_class"] == "high").astype(int)

    try:
        model = smf.mixedlm(
            f"{feat} ~ valence_bin",
            df_tmp,
            groups=df_tmp["participant"],
        ).fit(reml=True, disp=False)
        pval = model.pvalues["valence_bin"]
        tval = model.tvalues["valence_bin"]
        coef = model.params["valence_bin"]
    except Exception:
        pval = np.nan
        tval = np.nan
        coef = np.nan

    mean_low = vals_low.mean()
    mean_high = vals_high.mean()

    parts = feat.split("_", 1)  # e.g., "alpha_prefrontal"
    band_name = parts[0]
    region_name = parts[1] if len(parts) > 1 else "unknown"

    stat_rows.append({
        "feature": feat,
        "band": band_name,
        "region": region_name,
        "mean_low": mean_low,
        "mean_high": mean_high,
        "mean_diff": mean_high - mean_low,
        "coef": coef,
        "t_value": tval,
        "cohens_d": d,
        "abs_d": abs(d),
        "p_value": pval,
    })

df_stats = pd.DataFrame(stat_rows)

# Benjamini-Hochberg FDR correction
reject, p_adj, _, _ = multipletests(df_stats["p_value"].fillna(1.0).values, method="fdr_bh")
df_stats["p_adj_BH"] = p_adj
df_stats["sig_05"] = p_adj < 0.05
df_stats["sig_01"] = p_adj < 0.01

df_stats = df_stats.sort_values("abs_d", ascending=False).reset_index(drop=True)

print("\n Top features by |Cohen's d|:")
print(df_stats[["feature", "coef", "t_value", "cohens_d","p_value","p_adj_BH","sig_05"]].head(24).to_string(index=False))
print(f"\n Significant features (FDR-adjusted p < 0.05): {df_stats['sig_05'].sum()} / {len(df_stats)}")

df_stats.to_csv(RESULTS_DIR / "univariate_stats.csv", index=False)
print("Saved: univariate_stats.csv")

# =============================================================================
# FIGURE 1 — Cohen's d heatmap
# =============================================================================
 
band_order   = list(BANDS.keys())
region_order = list(REGIONS.keys())
 
d_matrix   = df_stats.pivot(index="band",   columns="region", values="cohens_d")
sig_matrix = df_stats.pivot(index="band",   columns="region", values="sig_05")
d_matrix   = d_matrix.reindex(index=band_order,   columns=region_order)
sig_matrix = sig_matrix.reindex(index=band_order, columns=region_order)
 
fig, ax = plt.subplots(figsize=(12, 6))
 
vabs = max(abs(d_matrix.values.min()), abs(d_matrix.values.max()), 0.1)
sns.heatmap(
    d_matrix,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    vmin=-vabs,
    vmax=vabs,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 13, "weight": "bold"},
    linewidths=0.8,
    linecolor="#e0e0e0",
    cbar_kws={"label": "Cohen's d", "shrink": 0.85},
)
 
# Significance stars
for r_idx, region in enumerate(region_order):
    for b_idx, band in enumerate(band_order):
        if sig_matrix.loc[band, region]:
            ax.text(
                r_idx + 0.5, b_idx + 0.22, "*",
                ha="center", va="center",
                fontsize=16, color="black", fontweight="bold",
            )
 
ax.set_title(
    "Cohen's d Effect Size per Frequency Band × Scalp Region\n"
    "High vs Low Valence; * = significant after BH FDR correction (p < 0.05)",
    fontsize=18, pad=14,
)
ax.set_xlabel("Scalp Region", labelpad=10)
ax.set_ylabel("Frequency Band", labelpad=10)
 
# Capitalise tick labels
ax.set_xticklabels([t.get_text().capitalize() for t in ax.get_xticklabels()], rotation=30, ha="right")
ax.set_yticklabels([t.get_text().capitalize() for t in ax.get_yticklabels()], rotation=0)
 
plt.tight_layout()
fig.savefig(FIG_DIR / "cohens_d_heatmap.png", dpi=300, bbox_inches="tight")
plt.close("all")
print("Saved: cohens_d_heatmap.png")


# =============================================================================
# FIGURE 2 — Topomaps
# =============================================================================

def grand_avg_topo(df_subset, sfreq, fmin, fmax, channel_names):
    all_power = []
    for _, row in df_subset.iterrows():
        try:
            eeg = loadmat(row["file"])["data"]
            if eeg.shape[0] != 128:
                continue
            nperseg = min(int(sfreq * 2), eeg.shape[1])
            freqs, psd = welch(eeg, fs=sfreq, nperseg=nperseg, axis=-1)
            freq_mask = (freqs >= fmin) & (freqs <= fmax)
            if not np.any(freq_mask):
                continue
            log_power = np.log(psd[:, freq_mask].mean(axis=1) + 1e-10)
            all_power.append(log_power)
        except:
            pass
    return np.mean(all_power, axis=0) if all_power else None

df_low_work  = df_work[df_work["valence_class"] == "low"].reset_index(drop=True)
df_high_work = df_work[df_work["valence_class"] == "high"].reset_index(drop=True)
 
print("\n[5] Plotting grand average topomaps ...")
 
n_bands = len(BANDS)
fig, axes = plt.subplots(3, n_bands, figsize=(6 * n_bands, 15))
 
row_labels = ["Low valence", "High valence", "High − Low"]
 
for col, (band_name, (fmin, fmax)) in enumerate(BANDS.items()):
    topo_low  = grand_avg_topo(df_low_work,  SFREQ, fmin, fmax, CHANNEL_NAMES)
    topo_high = grand_avg_topo(df_high_work, SFREQ, fmin, fmax, CHANNEL_NAMES)
 
    if topo_low is None or topo_high is None:
        continue
 
    topo_diff   = topo_high - topo_low
    shared_vmin = min(topo_low.min(), topo_high.min())
    shared_vmax = max(topo_low.max(), topo_high.max())
    diff_vabs   = max(abs(topo_diff).max(), 0.01)
 
    for row_idx, (topo, vlim, cmap) in enumerate(zip(
        [topo_low,  topo_high, topo_diff],
        [(shared_vmin, shared_vmax), (shared_vmin, shared_vmax), (-diff_vabs, diff_vabs)],
        ["Reds",    "Reds",    "RdBu_r"],
    )):
        ax = axes[row_idx, col]
        im, _ = mne.viz.plot_topomap(
            topo, info, axes=ax, show=False,
            cmap=cmap, vlim=vlim, contours=5,
        )
        cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.01)
        cbar.ax.tick_params(labelsize=24)
 
        if row_idx == 0:
            ax.set_title(
                f"{band_name.capitalize()}\n({fmin}–{fmax} Hz)",
                fontsize=32, fontweight="bold", pad=6,
            )
        if col == 0:
            ax.set_ylabel(row_labels[row_idx], fontsize=32, labelpad=6)
 
fig.suptitle(
    "Grand-Average Band Power Topomaps; Low vs High Valence\n"
    "and Difference Map (red = more power in high-valence; blue = less)",
    fontsize=42, y=0.98,
)
plt.tight_layout(w_pad=0.3, h_pad=0.5)
fig.savefig(FIG_DIR / "grand_avg_topomaps.png", dpi=300, bbox_inches="tight")
plt.close("all")
print("Saved: grand_avg_topomaps.png")

# =============================================================================
# 6. SECONDARY ANALYSIS: LEAK-FREE ABLATION CLASSIFICATION
# =============================================================================

print("\n[6] Running ablation classification (with nested feature selection) ...")

from sklearn.metrics import balanced_accuracy_score, f1_score

X_all = df_features[feature_cols].values
y_all = (df_features["valence_class"] == "high").astype(int).values
groups_all = df_features["participant"].values

gkf = GroupKFold(n_splits=5)

scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
}

def run_cv_static(name, feat_subset_cols, X_full, y, groups, cv):
    """Runs standard CV for predefined feature sets (no leakage risk)."""
    feat_idx = [feature_cols.index(c) for c in feat_subset_cols]
    X_sub = X_full[:, feat_idx]

    clf = Pipeline([
        ("scaler",  StandardScaler()),
        ("logreg",  LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42))
    ])

    res = cross_validate(clf, X_sub, y, cv=cv, groups=groups, scoring=scoring, return_train_score=False)
    return {
        "subset": name,
        "n_features": len(feat_subset_cols),
        "balanced_accuracy_mean": np.mean(res["test_balanced_accuracy"]),
        "balanced_accuracy_std": np.std(res["test_balanced_accuracy"]),
        "f1_macro_mean": np.mean(res["test_f1_macro"]),
        "f1_macro_std": np.std(res["test_f1_macro"]),
    }

ablation_results = []

# --- A. Evaluate Predefined/Static Subsets ---
ablation_results.append(run_cv_static("All features", feature_cols, X_all, y_all, groups_all, gkf))

for band_name in BANDS:
    band_feats = [c for c in feature_cols if c.startswith(f"{band_name}_")]
    if band_feats:
        ablation_results.append(run_cv_static(f"Band: {band_name}", band_feats, X_all, y_all, groups_all, gkf))

for region_name in REGIONS:
    region_feats = [c for c in feature_cols if c.endswith(f"_{region_name}")]
    if region_feats:
        ablation_results.append(run_cv_static(f"Region: {region_name}", region_feats, X_all, y_all, groups_all, gkf))


# --- B. Evaluate Data-Driven Subsets (STRICTLY NESTED) ---
# We loop manually through the splits to compute Cohen's d ONLY on the training folds.
for k in [1, 3, 5, 10]:
    fold_bal_acc = []
    fold_f1_macro = []
    
    for train_idx, test_idx in gkf.split(X_all, y_all, groups_all):
        X_train, X_test = X_all[train_idx], X_all[test_idx]
        y_train, y_test = y_all[train_idx], y_all[test_idx]
        
        # Calculate Cohen's d using ONLY training fold data to find top features
        low_idx = np.where(y_train == 0)[0]
        high_idx = np.where(y_train == 1)[0]
        
        train_abs_ds = []
        for f_idx in range(X_train.shape[1]):
            a = X_train[high_idx, f_idx]
            b = X_train[low_idx, f_idx]
            
            n_a, n_b = len(a), len(b)
            pooled_std = np.sqrt(
                ((n_a - 1) * np.std(a, ddof=1)**2 + (n_b - 1) * np.std(b, ddof=1)**2)
                / (n_a + n_b - 2)
            )
            d = (np.mean(a) - np.mean(b)) / (pooled_std + 1e-10)
            train_abs_ds.append(abs(d))
            
        # Select the top-k feature indices based on training absolute effect sizes
        top_k_indices = np.argsort(train_abs_ds)[::-1][:k]
        
        # Isolate those features for this fold's train and test matrices
        X_train_nested = X_train[:, top_k_indices]
        X_test_nested = X_test[:, top_k_indices]
        
        # Train and validate the model
        clf = Pipeline([
            ("scaler",  StandardScaler()),
            ("logreg",  LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42))
        ])
        
        clf.fit(X_train_nested, y_train)
        preds = clf.predict(X_test_nested)
        
        fold_bal_acc.append(balanced_accuracy_score(y_test, preds))
        fold_f1_macro.append(f1_score(y_test, preds, average="macro"))
        
    ablation_results.append({
        "subset": f"Top-{k} Cohen's d",
        "n_features": k,
        "balanced_accuracy_mean": np.mean(fold_bal_acc),
        "balanced_accuracy_std": np.std(fold_bal_acc),
        "f1_macro_mean": np.mean(fold_f1_macro),
        "f1_macro_std": np.std(fold_f1_macro),
    })

df_ablation = pd.DataFrame(ablation_results)
df_ablation = df_ablation.sort_values("balanced_accuracy_mean", ascending=False)

print("\n Ablation results:")
print(df_ablation[["subset", "n_features", "balanced_accuracy_mean", "balanced_accuracy_std", "f1_macro_mean", "f1_macro_std"]].to_string(index=False))

df_ablation.to_csv(RESULTS_DIR / "ablation_results.csv", index=False)
print("\n Saved: ablation_results.csv")


# =============================================================================
# FIGURE 3 — Ablation bar chart
# =============================================================================
 
print("\n[7] Plotting ablation results ...")
 
df_plot = df_ablation.copy()
 
# Colour by subset type
def get_color(name):
    if "All"    in name: return "#555555"
    if "Band:"  in name: return "#4C72B0"
    if "Region" in name: return "#DD8452"
    if "Top-"   in name: return "#55A868"
    return "#888888"
 
colors = [get_color(s) for s in df_plot["subset"]]
 
fig, ax = plt.subplots(figsize=(13, 7))
y_pos = np.arange(len(df_plot))
 
ax.barh(
    y_pos,
    df_plot["balanced_accuracy_mean"],
    xerr=df_plot["balanced_accuracy_std"],
    color=colors,
    edgecolor="white",
    capsize=5,
    height=0.65,
    error_kw={"elinewidth": 1.5, "ecolor": "#333333"},
)
 
# Annotate each bar with its mean value
for i, (mean, std) in enumerate(zip(
    df_plot["balanced_accuracy_mean"], df_plot["balanced_accuracy_std"]
)):
    ax.text(
        mean + df_plot["balanced_accuracy_std"].iloc[i] + 0.003,
        i,
        f"{mean:.3f}",
        va="center", ha="left",
        fontsize=14, color="#333333",
    )
 
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot["subset"], fontsize=16)
ax.axvline(0.5, color="black", linewidth=1.2, linestyle="--", label="Chance level (0.50)")
 
ax.set_xlabel("Balanced Accuracy (mean ± SD across folds)", fontsize=18, labelpad=10)
ax.set_title(
    "Ablation Classification; Balanced Accuracy by Feature Subset\n"
    "Blue = freq. bands; Orange = scalp regions; Green = Cohen's d features; Grey = all features",
    fontsize=18, pad=8,
)
 
ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.025))
ax.tick_params(axis="x", which="both", labelsize=18)
ax.set_xlim(0.38, max(df_plot["balanced_accuracy_mean"] + df_plot["balanced_accuracy_std"]) + 0.07)
 
ax.legend(fontsize=16, framealpha=0.4)
plt.tight_layout()
fig.savefig(FIG_DIR / "ablation_balanced_accuracy.png", dpi=200, bbox_inches="tight")
plt.close("all")
print("Saved: ablation_balanced_accuracy.png")


# =============================================================================
# 8. COMBINED SUMMARY PRINTOUT
# =============================================================================

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print("\n--- TOP BAND×REGION COMBINATIONS BY EFFECT SIZE ---")
print(df_stats[["feature","cohens_d","p_adj_BH","sig_05"]].head(10).to_string(index=False))

print("\n--- BAND-LEVEL MEAN |COHEN'S D| ---")
band_summary = (
    df_stats.groupby("band")["abs_d"]
    .agg(["mean","max"])
    .rename(columns={"mean":"mean_|d|","max":"max_|d|"})
    .sort_values("mean_|d|", ascending=False)
)
print(band_summary.to_string())

print("\n--- REGION-LEVEL MEAN |COHEN'S D| ---")
region_summary = (df_stats.groupby("region")["abs_d"]
    .agg(["mean","max"])
    .rename(columns={"mean":"mean_|d|","max":"max_|d|"})
    .sort_values("mean_|d|", ascending=False)
)
print(region_summary.to_string())

print("\n--- BEST ABLATION MODEL ---")
best_ablation = df_ablation[~df_ablation["subset"].str.contains("Dummy")].iloc[0]
print(f"Subset: {best_ablation['subset']}")
print(f"Balanced accuracy: {best_ablation['balanced_accuracy_mean']:.4f} ± {best_ablation['balanced_accuracy_std']:.4f}")
print(f"F1 macro: {best_ablation['f1_macro_mean']:.4f} ± {best_ablation['f1_macro_std']:.4f}")

print(f"\nAll outputs successfully saved to: {RESULTS_DIR}")
print("=" * 70)

CONFIG LOADED

[1] Extracting band×region features from .mat files ...
Trials (clicks) filtered for analysis: 453
Class balance:
valence_class
low     321
high    132
Final feature matrix: 453 rows x 24 features
Saved: trial_band_region_features.csv

[2] Running univariate statistics ...

 Top features by |Cohen's d|:
         feature      coef   t_value  cohens_d      p_value     p_adj_BH  sig_05
 gamma_occipital  0.158675  4.667619  0.397079 3.047106e-06 2.437685e-05    True
  gamma_parietal  0.154374  5.450470  0.357391 5.023680e-08 6.028416e-07    True
  gamma_temporal  0.251871  5.462570  0.240492 4.692911e-08 6.028416e-07    True
theta_prefrontal  0.011342  0.434084 -0.192811 6.642277e-01 7.591174e-01   False
  beta_occipital  0.041258  1.854212  0.185679 6.370889e-02 1.529013e-01   False
   gamma_frontal  0.139168  4.536277  0.161048 5.725591e-06 3.435355e-05    True
 beta_prefrontal  0.010666  0.488235 -0.139206 6.253833e-01 7.591174e-01   False
   beta_temporal  0.120437  3.81